# Real-Time Synthetic Data Streamer
Continuously generates synthetic fact-table records matching the same schema as the static data loaded into this lakehouse, and publishes them to a **Fabric Eventstream** custom endpoint (Event Hub–compatible).

**How to get the connection details:**
1. Open your Eventstream in Fabric → add a **Custom Endpoint** source.
2. Copy the **Connection string-primary key** and the **Event hub name**.
3. Paste them into the configuration cell below.

Parameters control how long the stream runs and the emission rate. Run all cells to start streaming; the final cell blocks until the duration elapses.

In [ ]:
# ── Configuration ───────────────────────────────────────────────────────────
# Eventstream custom endpoint (Event Hub compatible)
EVENTSTREAM_CONNECTION_STRING = "{{EVENTSTREAM_CONNECTION_STRING}}"  # Endpoint=sb://...;SharedAccessKeyName=...;SharedAccessKey=...;EntityPath=...
EVENTSTREAM_ENTITY_PATH = "{{EVENTSTREAM_ENTITY_PATH}}"  # Event hub name, e.g. "es_xxx"

# Stream shape
DURATION_MINUTES = {{DURATION_MINUTES}}       # how long to keep emitting (e.g., 10)
EVENTS_PER_SECOND = {{EVENTS_PER_SECOND}}     # emission rate (e.g., 5)
BATCH_SIZE = {{BATCH_SIZE}}                   # events flushed per batch (e.g., 20)

# Schema binding — tables this notebook references
FACT_TABLE_NAME = "{{FACT_TABLE_NAME}}"       # fact table whose schema we mirror
LAKEHOUSE_NAME = "{{LAKEHOUSE_NAME}}"         # display-only
DIMENSION_TABLES = {{DIMENSION_TABLES}}       # Python list of dim tables to sample FKs from, e.g. ["passengers","flights"]
# ───────────────────────────────────────────────────────────────────────

# Install the Event Hubs client (idempotent, ~5-10s on first run)
%pip install --quiet azure-eventhub

In [ ]:
import random
import time
import json
import uuid
from datetime import datetime, timezone
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# Load every dimension once into memory as a list of dicts so we can sample FKs
# and realistic categorical values without hitting the lakehouse on every event.
dim_cache = {}
for dim in DIMENSION_TABLES:
    rows = spark.table(dim).limit(50000).toPandas().to_dict(orient="records")
    dim_cache[dim] = rows
    print(f"  cached {dim}: {len(rows):,} rows")

print(f"\nFact schema target: {FACT_TABLE_NAME}")
spark.table(FACT_TABLE_NAME).printSchema()

In [ ]:
# ── Record generator ─────────────────────────────────────────────────────────────
# Produces one synthetic event that matches the fact-table schema.
# Sampled FKs come from dim_cache; non-FK columns use realistic randomized values.
# The agent fills the body based on the schema it designed.
_event_counter = {"n": 0}

def generate_event() -> dict:
    _event_counter["n"] += 1
    event = {
        # Primary key (monotonic, unique per run)
        "{{PK_COLUMN}}": int(time.time() * 1000) * 1000 + _event_counter["n"],
        # Event timestamp — always "now" so downstream consumers see fresh data
        "event_timestamp": datetime.now(timezone.utc).isoformat(),

        # ↓↓↓  RECORD BODY — generated by the agent from the fact schema  ↓↓↓
        {{RECORD_GENERATOR_BODY}}
        # ↑↑↑  RECORD BODY — generated by the agent from the fact schema  ↑↑↑
    }
    return event

In [ ]:
# ── Streaming loop ──────────────────────────────────────────────────────────────────
from azure.eventhub import EventHubProducerClient, EventData

producer = EventHubProducerClient.from_connection_string(
    conn_str=EVENTSTREAM_CONNECTION_STRING,
    eventhub_name=EVENTSTREAM_ENTITY_PATH,
)

total_duration_s = DURATION_MINUTES * 60
interval_s = 1.0 / EVENTS_PER_SECOND if EVENTS_PER_SECOND > 0 else 1.0

start = time.time()
sent = 0
batch = producer.create_batch()

print(f"Streaming to '{EVENTSTREAM_ENTITY_PATH}' for {DURATION_MINUTES} min "
      f"at ~{EVENTS_PER_SECOND} events/s (batch={BATCH_SIZE})")

try:
    while (time.time() - start) < total_duration_s:
        event = generate_event()
        ed = EventData(json.dumps(event, default=str))
        try:
            batch.add(ed)
        except ValueError:
            # Batch full → flush and start a new one
            producer.send_batch(batch)
            sent += len(batch)
            batch = producer.create_batch()
            batch.add(ed)

        if len(batch) >= BATCH_SIZE:
            producer.send_batch(batch)
            sent += len(batch)
            batch = producer.create_batch()

        elapsed = time.time() - start
        if sent > 0 and sent % (EVENTS_PER_SECOND * 10) == 0:
            print(f"  t+{int(elapsed):4d}s  sent={sent:,}")

        time.sleep(interval_s)

    # Flush remainder
    if len(batch) > 0:
        producer.send_batch(batch)
        sent += len(batch)

finally:
    producer.close()

print(f"\nDone. Sent {sent:,} events in {int(time.time() - start)}s.")